# BUN Rule Validation

**Source script:** `bun_rule_validation.py`

Notebook mirror for submission traceability.

In [ ]:
from __future__ import annotations

import argparse
import re
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parent
DEFAULT_INPUT = ROOT / "outputs" / "imputed_dataset_enriched_with_clinical_states.csv"
DEFAULT_RULES = ROOT / "outputs" / "eda_v3" / "apriori_refined_final" / "severity_rules_clinical_thresholds.csv"
DEFAULT_OUTDIR = ROOT / "outputs" / "eda_v3" / "clinical_threshold_extension" / "bun_rule_validation"

ANTECEDENT_ITEMS = [
    "precipitation_hours_mean_7d_low",
    "precipitation_hours_peak_14d_low",
    "surface_pressure_max_day1_high",
]


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Validate strongest BUN_increased weather Apriori rule.")
    parser.add_argument("--input", type=Path, default=DEFAULT_INPUT, help="Input dataset with clinical states")
    parser.add_argument("--rules", type=Path, default=DEFAULT_RULES, help="Severity rules CSV")
    parser.add_argument("--outdir", type=Path, default=DEFAULT_OUTDIR, help="Output directory")
    return parser.parse_args()


def normalize_name(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(text).lower())


def resolve_input_path(path: Path, fallbacks: List[Path], label: str) -> Path:
    if path.exists():
        return path
    for f in fallbacks:
        if f.exists():
            print(f"[info] {label} not found at {path}; using fallback: {f}")
            return f
    raise FileNotFoundError(f"{label} not found. Checked: {[str(path)] + [str(x) for x in fallbacks]}")


def find_col(df: pd.DataFrame, wanted_normalized: str) -> str:
    for c in df.columns:
        if normalize_name(c) == wanted_normalized:
            return c
    raise KeyError(f"Column not found for normalized name: {wanted_normalized}")


def compute_quantile_flag(series: pd.Series, mode: str, q_low: float, q_high: float) -> Tuple[pd.Series, float]:
    s = pd.to_numeric(series, errors="coerce")
    valid = s.dropna()
    if valid.empty:
        return pd.Series(False, index=series.index), np.nan
    if mode == "low":
        q = float(valid.quantile(q_low))
        return (s <= q).fillna(False), q
    q = float(valid.quantile(q_high))
    return (s >= q).fillna(False), q


def compute_rule_metrics(ante_mask: pd.Series, y: pd.Series) -> Dict[str, float]:
    a = ante_mask.fillna(False).astype(bool)
    yy = y.fillna(False).astype(bool)
    n = len(a)
    if n == 0:
        return {"support": np.nan, "support_count": 0, "confidence": np.nan, "lift": np.nan, "antecedent_prevalence": np.nan}
    support = float((a & yy).mean())
    support_count = int((a & yy).sum())
    ante_prev = float(a.mean())
    baseline = float(yy.mean())
    confidence = float(support / ante_prev) if ante_prev > 0 else np.nan
    lift = float(confidence / baseline) if baseline > 0 and np.isfinite(confidence) else np.nan
    return {
        "support": support,
        "support_count": support_count,
        "confidence": confidence,
        "lift": lift,
        "antecedent_prevalence": ante_prev,
        "baseline_prevalence": baseline,
    }


def parse_antecedent_set(text: str) -> set[str]:
    if not isinstance(text, str) or not text.strip():
        return set()
    return set([x.strip() for x in text.split(",") if x.strip()])


def format_pct(x: float) -> str:
    if pd.isna(x):
        return "NA"
    return f"{100.0 * x:.2f}%"


def main() -> None:
    args = parse_args()
    args.outdir.mkdir(parents=True, exist_ok=True)

    dataset_path = resolve_input_path(
        args.input,
        fallbacks=[ROOT / "outputs" / "eda_v3" / "clinical_threshold_extension" / "imputed_dataset_enriched_with_clinical_states.csv"],
        label="dataset",
    )
    rules_path = resolve_input_path(
        args.rules,
        fallbacks=[
            ROOT / "outputs" / "eda_v3" / "clinical_threshold_extension" / "severity_rules_clinical_thresholds.csv",
            ROOT / "outputs" / "eda_v3" / "apriori_refined_final" / "severity_rules.csv",
        ],
        label="rules file",
    )

    df = pd.read_csv(dataset_path)
    rules_df = pd.read_csv(rules_path)


    col_prec_mean7 = find_col(df, "precipitationhoursmean7d")
    col_prec_peak14 = find_col(df, "precipitationhourspeak14d")
    col_press_day1 = find_col(df, "surfacepressuremaxday1")

    if "BUN_increased" in df.columns:
        y = df["BUN_increased"].fillna(False).astype(bool)
    elif "bun_state" in df.columns:
        y = df["bun_state"].astype(str).eq("increased")
    else:
        raise KeyError("Neither BUN_increased nor bun_state exists in input dataset.")


    f1, q1 = compute_quantile_flag(df[col_prec_mean7], mode="low", q_low=0.25, q_high=0.75)
    f2, q2 = compute_quantile_flag(df[col_prec_peak14], mode="low", q_low=0.25, q_high=0.75)
    f3, q3 = compute_quantile_flag(df[col_press_day1], mode="high", q_low=0.25, q_high=0.75)
    full_mask = f1 & f2 & f3

    full_metrics = compute_rule_metrics(full_mask, y)
    n_total = len(df)
    bun_prev = float(y.mean())

    antecedent_prevalence = {
        ANTECEDENT_ITEMS[0]: float(f1.mean()),
        ANTECEDENT_ITEMS[1]: float(f2.mean()),
        ANTECEDENT_ITEMS[2]: float(f3.mean()),
        "full_conjunction": float(full_mask.mean()),
    }


    apriori_conf = np.nan
    apriori_lift = np.nan
    apriori_support = np.nan
    if not rules_df.empty:
        if "severity_marker" in rules_df.columns:
            sub = rules_df[rules_df["severity_marker"] == "BUN_increased"].copy()
        elif "consequent_name" in rules_df.columns:
            sub = rules_df[rules_df["consequent_name"] == "BUN_increased"].copy()
        else:
            sub = pd.DataFrame()
        if not sub.empty and "antecedents" in sub.columns:
            target_set = set(ANTECEDENT_ITEMS)
            sub["ante_set"] = sub["antecedents"].map(parse_antecedent_set)
            hit = sub[sub["ante_set"].map(lambda s: s == target_set)]
            if not hit.empty:
                row = hit.sort_values(["lift", "confidence"], ascending=[False, False]).iloc[0]
                apriori_conf = float(row.get("confidence", np.nan))
                apriori_lift = float(row.get("lift", np.nan))
                apriori_support = float(row.get("support", np.nan))


    variants = [
        ("full_rule", [f1, f2, f3]),
        ("drop_precipitation_hours_peak_14d_low", [f1, f3]),
        ("drop_precipitation_hours_mean_7d_low", [f2, f3]),
        ("drop_surface_pressure_max_day1_high", [f1, f2]),
    ]
    abl_rows = []
    for name, masks in variants:
        m = masks[0]
        for mm in masks[1:]:
            m = m & mm
        metrics = compute_rule_metrics(m, y)
        abl_rows.append(
            {
                "variant": name,
                "antecedent_count": len(masks),
                "support": metrics["support"],
                "support_count": metrics["support_count"],
                "confidence": metrics["confidence"],
                "lift": metrics["lift"],
                "baseline_prevalence": metrics["baseline_prevalence"],
                "antecedent_prevalence": metrics["antecedent_prevalence"],
            }
        )
    ablation_df = pd.DataFrame(abl_rows)
    ablation_df.to_csv(args.outdir / "bun_rule_ablation_table.csv", index=False)


    f1_20, q1_20 = compute_quantile_flag(df[col_prec_mean7], mode="low", q_low=0.20, q_high=0.80)
    f2_20, q2_20 = compute_quantile_flag(df[col_prec_peak14], mode="low", q_low=0.20, q_high=0.80)
    f3_20, q3_20 = compute_quantile_flag(df[col_press_day1], mode="high", q_low=0.20, q_high=0.80)
    m20 = f1_20 & f2_20 & f3_20
    sens = compute_rule_metrics(m20, y)
    sens_df = pd.DataFrame(
        [
            {
                "rule": "full_rule_20_80",
                "q_low": 0.20,
                "q_high": 0.80,
                "precipitation_hours_mean_7d_threshold_low": q1_20,
                "precipitation_hours_peak_14d_threshold_low": q2_20,
                "surface_pressure_max_day1_threshold_high": q3_20,
                "support": sens["support"],
                "support_count": sens["support_count"],
                "confidence": sens["confidence"],
                "lift": sens["lift"],
                "baseline_prevalence": sens["baseline_prevalence"],
                "antecedent_prevalence": sens["antecedent_prevalence"],
            }
        ]
    )
    sens_df.to_csv(args.outdir / "bun_rule_sensitivity_20_80.csv", index=False)


    outcome_col = None
    for c in ["group", "diagnosis_group", "outcome_group", "outcome"]:
        if c in df.columns:
            outcome_col = c
            break
    if outcome_col is None:
        raise KeyError("No outcome column found for disease distribution.")

    disease = df[outcome_col].astype(str)
    overall = disease.value_counts(dropna=False).rename("overall_n")
    overall.index.name = "disease"
    subset = disease[full_mask].value_counts(dropna=False).rename("rule_subset_n")
    subset.index.name = "disease"
    dis_df = pd.concat([overall, subset], axis=1).fillna(0).reset_index()
    if "disease" not in dis_df.columns:
        first_col = dis_df.columns[0]
        dis_df = dis_df.rename(columns={first_col: "disease"})
    dis_df["overall_n"] = dis_df["overall_n"].astype(int)
    dis_df["rule_subset_n"] = dis_df["rule_subset_n"].astype(int)
    dis_df["overall_pct"] = dis_df["overall_n"] / max(1, n_total)
    dis_df["rule_subset_pct"] = dis_df["rule_subset_n"] / max(1, int(full_mask.sum()))
    dis_df["subset_vs_overall_ratio"] = np.where(dis_df["overall_pct"] > 0, dis_df["rule_subset_pct"] / dis_df["overall_pct"], np.nan)
    dis_df = dis_df.sort_values("rule_subset_n", ascending=False).reset_index(drop=True)
    dis_df.to_csv(args.outdir / "bun_rule_disease_distribution.csv", index=False)


    strongest_ablation = ablation_df[ablation_df["variant"] != "full_rule"].sort_values("lift", ascending=False).head(1)
    strongest_ablation_lift = float(strongest_ablation.iloc[0]["lift"]) if not strongest_ablation.empty else np.nan
    sens_lift = float(sens["lift"])
    full_lift = float(full_metrics["lift"])

    ablation_survives = bool((ablation_df[ablation_df["variant"] != "full_rule"]["lift"] > 1.5).all()) if len(ablation_df) > 1 else False
    sens_survives = bool(sens_lift > 1.5)
    if full_lift > 1.5 and ablation_survives and sens_survives:
        robustness = "Stable and robust"
    elif full_lift > 1.5 and (ablation_survives or sens_survives):
        robustness = "Moderately sensitive"
    else:
        robustness = "Threshold-dependent / unstable"

    md_lines: List[str] = []
    md_lines.append("# BUN Rule Validation Summary")
    md_lines.append("")
    md_lines.append("## Baseline")
    md_lines.append(f"- Total N: {n_total}")
    md_lines.append(f"- BUN_increased prevalence: {format_pct(bun_prev)} ({int(y.sum())}/{n_total})")
    md_lines.append(f"- Antecedent prevalence: {ANTECEDENT_ITEMS[0]}={format_pct(antecedent_prevalence[ANTECEDENT_ITEMS[0]])}, {ANTECEDENT_ITEMS[1]}={format_pct(antecedent_prevalence[ANTECEDENT_ITEMS[1]])}, {ANTECEDENT_ITEMS[2]}={format_pct(antecedent_prevalence[ANTECEDENT_ITEMS[2]])}")
    md_lines.append(f"- Full conjunction prevalence: {format_pct(antecedent_prevalence['full_conjunction'])} ({int(full_mask.sum())}/{n_total})")
    md_lines.append("")
    md_lines.append("## Full Rule Metrics")
    md_lines.append(f"- Support: {full_metrics['support']:.4f} (count={full_metrics['support_count']})")
    md_lines.append(f"- Confidence: {full_metrics['confidence']:.4f}")
    md_lines.append(f"- Lift: {full_metrics['lift']:.4f}")
    if np.isfinite(apriori_conf):
        md_lines.append(f"- Apriori confidence reference: {apriori_conf:.4f}")
        md_lines.append(f"- Apriori lift reference: {apriori_lift:.4f}")
        md_lines.append(f"- Apriori support reference: {apriori_support:.4f}")
    md_lines.append("")
    md_lines.append("## Ablation Survival (>1.5 lift)")
    md_lines.append(f"- Strongest ablation lift: {strongest_ablation_lift:.4f}")
    md_lines.append(f"- Survives ablation criterion: {'yes' if ablation_survives else 'no'}")
    md_lines.append("")
    md_lines.append("## Discretization Sensitivity (20/80)")
    md_lines.append(f"- 20/80 lift: {sens_lift:.4f}")
    md_lines.append(f"- Survives sensitivity criterion (>1.5): {'yes' if sens_survives else 'no'}")
    md_lines.append("")
    md_lines.append("## Disease Composition in Rule-Covered Subset")
    for _, r in dis_df.iterrows():
        md_lines.append(
            f"- {r['disease']}: subset {int(r['rule_subset_n'])} ({format_pct(float(r['rule_subset_pct']))}), "
            f"overall {int(r['overall_n'])} ({format_pct(float(r['overall_pct']))}), "
            f"ratio {float(r['subset_vs_overall_ratio']):.2f}"
        )
    md_lines.append("")
    md_lines.append("## Interpretation")
    md_lines.append(f"- Robustness classification: **{robustness}**.")
    md_lines.append("- This is a post-hoc stability check of one association rule; it remains exploratory and non-causal.")

    (args.outdir / "bun_rule_validation_summary.md").write_text("\n".join(md_lines), encoding="utf-8")


    baseline_df = pd.DataFrame(
        [
            {"metric": "N_total", "value": n_total},
            {"metric": "BUN_increased_count", "value": int(y.sum())},
            {"metric": "BUN_increased_prevalence", "value": bun_prev},
            {"metric": f"{ANTECEDENT_ITEMS[0]}_prevalence", "value": antecedent_prevalence[ANTECEDENT_ITEMS[0]]},
            {"metric": f"{ANTECEDENT_ITEMS[1]}_prevalence", "value": antecedent_prevalence[ANTECEDENT_ITEMS[1]]},
            {"metric": f"{ANTECEDENT_ITEMS[2]}_prevalence", "value": antecedent_prevalence[ANTECEDENT_ITEMS[2]]},
            {"metric": "full_conjunction_prevalence", "value": antecedent_prevalence["full_conjunction"]},
            {"metric": "full_support", "value": full_metrics["support"]},
            {"metric": "full_confidence", "value": full_metrics["confidence"]},
            {"metric": "full_lift", "value": full_metrics["lift"]},
            {"metric": "full_support_count", "value": full_metrics["support_count"]},
            {"metric": "threshold_q25_precipitation_hours_mean_7d", "value": q1},
            {"metric": "threshold_q25_precipitation_hours_peak_14d", "value": q2},
            {"metric": "threshold_q75_surface_pressure_max_day1", "value": q3},
        ]
    )
    baseline_df.to_csv(args.outdir / "bun_rule_baseline_stats.csv", index=False)

    print("=== BUN Rule Validation ===")
    print(f"Full rule lift: {full_lift:.4f}")
    print(f"Strongest ablation variant lift: {strongest_ablation_lift:.4f}")
    print(f"20/80 lift: {sens_lift:.4f}")
    print(f"Final robustness classification: {robustness}")


if __name__ == "__main__":
    main()
